# Merge Clean Water Quality and Rainfall (ML Ready)

This notebook joins the two cleaned datasets into the single model ready table
described in the project overview. The end product is one row per beach water
sample, enriched with antecedent rainfall features, ready to train the model.

Inputs:

- `resources/clean_data/clean_water_quality_data.csv` (one row per beach sample
  with the `unsafe` label)
- `resources/clean_data/clean_rainfall_data.csv` (one row per station per day)

Output:

- `resources/clean_data/master_dataset.csv`

## Why these merge decisions

The research question is whether rainfall in the days leading up to a sample can
predict unsafe swimming. That goal drives every choice below.

1. Nearest station by geographic distance. Each beach is permanently assigned to
   the single closest rain gauge using great circle distance on lat / lon. A beach
   has no rain gauge of its own, so the closest gauge is the best available proxy
   for the rain that actually fell on that shoreline.

2. Antecedent only rainfall windows. Every rainfall feature is built from days
   strictly before the sample date. We never include rainfall from the sample day
   itself. Including same day rain would leak information that would not be known
   at prediction time and would inflate the model artificially.

3. Pre-compute features per station then left join on date. The lag features are
   computed once per station per day, then the beach samples (left) are joined to
   that feature table. A left join keeps every valid beach sample and attaches its
   matching rainfall context without dropping samples just because of join order.

4. Drop rows whose lag window cannot be filled. If the assigned station is missing
   any day inside a feature window, that feature cannot be trusted, so the sample
   is dropped rather than fed a guessed value. This protects the model from
   training on fabricated rainfall history.


In [ ]:
## Imports and configuration

# numpy powers the vectorized distance math, pandas does the table work. The path
# constants point at the two clean inputs and the master output. The lag windows
# list the day spans the model will learn from.
import os

import numpy as np
import pandas as pd

# Notebook lives in backend/data_scripts/ so go up two levels to the project root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CLEAN_DIR = os.path.join(PROJECT_ROOT, "resources", "clean_data")
WQ_PATH = os.path.join(CLEAN_DIR, "clean_water_quality_data.csv")
RAIN_PATH = os.path.join(CLEAN_DIR, "clean_rainfall_data.csv")
OUTPUT_PATH = os.path.join(CLEAN_DIR, "master_dataset.csv")

# A day counts as dry when it had zero rainfall. Used for days_since_rain.
DRY_DAY_MM = 0.0


In [ ]:
## Load both clean datasets

# station_id and location_id are read as text so identifiers stay exact. Dates are
# parsed to real datetimes for the rolling window math and the join.
wq = pd.read_csv(WQ_PATH, dtype={"location_id": str})
wq["date"] = pd.to_datetime(wq["date"])

rain = pd.read_csv(RAIN_PATH, dtype={"station_id": str})
rain["date"] = pd.to_datetime(rain["date"])

print("Water quality:", wq.shape, "beaches:", wq["location_id"].nunique())
print("Rainfall:", rain.shape, "stations:", rain["station_id"].nunique())


In [ ]:
## Assign each beach to its nearest rain gauge

# Each beach has one fixed location, so we work with the unique beach coordinates
# and the unique station coordinates rather than the full sample table. For every
# beach we compute the great circle (haversine) distance to every station and keep
# the closest one. Great circle distance is the correct measure on the curved earth
# surface and avoids the distortion a plain lat / lon subtraction would introduce.
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0  # earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return r * 2 * np.arcsin(np.sqrt(a))


beaches = wq[["location_id", "latitude", "longitude"]].drop_duplicates("location_id")
stations = rain[["station_id", "lat", "lon"]].drop_duplicates("station_id")

# Broadcast every beach against every station to get a distance matrix.
b_lat = beaches["latitude"].to_numpy()[:, None]
b_lon = beaches["longitude"].to_numpy()[:, None]
s_lat = stations["lat"].to_numpy()[None, :]
s_lon = stations["lon"].to_numpy()[None, :]

dist = haversine_km(b_lat, b_lon, s_lat, s_lon)
nearest_idx = dist.argmin(axis=1)

beaches = beaches.assign(
    nearest_station_id=stations["station_id"].to_numpy()[nearest_idx],
    nearest_station_km=dist.min(axis=1).round(3),
)

beach_to_station = beaches[["location_id", "nearest_station_id"]]
print("Beaches assigned:", len(beaches))
print("Median distance to gauge (km):", round(beaches["nearest_station_km"].median(), 2))
beaches.head()


In [ ]:
## Build a continuous daily grid per station

# The rainfall windows are measured in calendar days, so a rolling window only lines
# up with real days if every station has one row per day. We reindex each station to
# a complete daily range. Days that were dropped during cleaning come back as NaN
# here, which is exactly what makes an incomplete window resolve to NaN later and get
# dropped. We only need the station, date, and rainfall columns for the feature math.
rain_small = rain[["station_id", "date", "rainfall_mm"]].sort_values(
    ["station_id", "date"]
)

grids = []
for station_id, group in rain_small.groupby("station_id"):
    full_index = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    g = group.set_index("date")[["rainfall_mm"]].reindex(full_index)
    g.index.name = "date"
    g["station_id"] = station_id
    grids.append(g.reset_index())

grid = pd.concat(grids, ignore_index=True).sort_values(["station_id", "date"])
grid = grid.reset_index(drop=True)

print("Daily grid rows:", len(grid))
print("Missing days exposed:", int(grid["rainfall_mm"].isna().sum()))


In [ ]:
## Engineer the antecedent rainfall features

# Every feature is built from an antecedent series: rainfall shifted forward one day
# within each station. The shift is the leakage guard. It means the value lined up
# with sample date D is the rainfall from D-1, so the sample day itself is never used
# as an input. min_periods equal to the window forces the whole window to be present,
# otherwise the feature is NaN and the row is later dropped.
grid["antecedent_mm"] = grid.groupby("station_id")["rainfall_mm"].shift(1)
roll = grid.groupby("station_id")["antecedent_mm"]

# Summed rainfall over the day spans the model learns from.
grid["rain_24hr"] = grid["antecedent_mm"]
grid["rain_48hr"] = roll.rolling(2, min_periods=2).sum().reset_index(level=0, drop=True)
grid["rain_72hr"] = roll.rolling(3, min_periods=3).sum().reset_index(level=0, drop=True)
grid["rain_7day"] = roll.rolling(7, min_periods=7).sum().reset_index(level=0, drop=True)

# Heaviest single day inside the prior 3 days, captures short intense storms.
grid["max_rain_3day"] = roll.rolling(3, min_periods=3).max().reset_index(
    level=0, drop=True
)

# days_since_rain: consecutive dry days immediately before the sample date. A dry day
# is one with zero antecedent rainfall. We tag dry days, start a new block at every
# wet day, then count consecutive dry days within each block.
grid["is_dry"] = (grid["antecedent_mm"] <= DRY_DAY_MM).astype("Int64")
wet_flag = (grid["is_dry"] == 0).astype(int)
grid["block"] = wet_flag.groupby(grid["station_id"]).cumsum()
grid["days_since_rain"] = (
    grid["is_dry"].groupby([grid["station_id"], grid["block"]]).cumsum()
)

feature_cols = [
    "rain_24hr",
    "rain_48hr",
    "rain_72hr",
    "rain_7day",
    "max_rain_3day",
    "days_since_rain",
]
features = grid[["station_id", "date"] + feature_cols]

print("Feature rows:", len(features))
features.head(10)


In [ ]:
## Join beach samples to their station rainfall features

# Step 1 attaches each beach sample to its assigned nearest station id. Step 2 left
# joins the sample table to the pre-computed feature table on station and date. A
# left join keeps every beach sample and pulls in the matching rainfall context. If
# the assigned station has no row for that date, or its window was incomplete, the
# feature columns come back NaN and get cleaned up in the next step.
merged = wq.merge(beach_to_station, on="location_id", how="left")
merged = merged.merge(
    features,
    left_on=["nearest_station_id", "date"],
    right_on=["station_id", "date"],
    how="left",
)
merged = merged.drop(columns=["station_id"])

print("Merged shape:", merged.shape)
print("Samples missing rain_7day:", int(merged["rain_7day"].isna().sum()))
merged.head()


In [ ]:
## Drop incomplete windows and finalize the schema

# Any sample still missing a rainfall feature has an unusable lag window (early dates
# before 7 days of history exist, or a station gap). Those rows are dropped so the
# model never sees a fabricated value. We then order the columns to match the master
# dataset schema in the project overview. date is written back as plain YYYY-MM-DD.
feature_cols = [
    "rain_24hr",
    "rain_48hr",
    "rain_72hr",
    "rain_7day",
    "max_rain_3day",
    "days_since_rain",
]

before = len(merged)
master = merged.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Rows: kept {len(master)} of {before}")

# days_since_rain is a whole number of days; store it as an integer.
master["days_since_rain"] = master["days_since_rain"].astype(int)
master["date"] = master["date"].dt.strftime("%Y-%m-%d")

master = master[
    [
        "date",
        "location_id",
        "location_name",
        "latitude",
        "longitude",
        "enterococcus",
        "unsafe",
        "nearest_station_id",
        "rain_24hr",
        "rain_48hr",
        "rain_72hr",
        "rain_7day",
        "days_since_rain",
        "max_rain_3day",
        "month",
    ]
]
master = master.sort_values(["location_id", "date"]).reset_index(drop=True)

print("Master shape:", master.shape)
master.head()


In [ ]:
## Export the master dataset

# Write the model ready table. One row per beach sample, with the unsafe label and
# the antecedent rainfall features. This is the file the model training step loads.
master.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(master):,} rows to {OUTPUT_PATH}")
print("Beaches:", master["location_id"].nunique())
print("Date range:", master["date"].min(), "to", master["date"].max())
print("Unsafe rate:", round(master["unsafe"].mean(), 4))
